In [77]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import pickle
import json
import re
from datetime import datetime
from collections import Counter
from math import log

In [78]:
# ===============================
# 1. LOAD & PREPROCESSING DATA
# ===============================

# Function to load data
def load_data():
    # Dalam implementasi nyata, akan mengambil data dari database
    articles_df = pd.read_csv('articles.csv')
    likes_df = pd.read_csv('likes.csv')
    comments_df = pd.read_csv('comments.csv')
    return articles_df, likes_df, comments_df


In [79]:
# Function to preprocess data
def preprocess_data(articles_df, likes_df, comments_df):
    # Calculate likes per article
    article_likes = likes_df.groupby('article_id').size().reset_index(name='likes_count')
    
    # Calculate comments per article
    article_comments = comments_df.groupby('article_id').size().reset_index(name='comments_count')
    
    # Combine data with articles
    articles_enriched = articles_df.copy()
    articles_enriched['UUID'] = articles_enriched['UUID'].astype(str)
    
    # Add likes count
    articles_enriched = articles_enriched.merge(article_likes, left_on='UUID', right_on='article_id', how='left')
    articles_enriched['likes_count'] = articles_enriched['likes_count'].fillna(0)
    
    # Add comments count
    articles_enriched = articles_enriched.merge(article_comments, left_on='UUID', right_on='article_id', how='left')
    articles_enriched['comments_count'] = articles_enriched['comments_count'].fillna(0)
    
    # Clean up
    articles_enriched = articles_enriched.drop(['article_id_x', 'article_id_y'], axis=1, errors='ignore')
    
    # Extract features from article title using Text Processing
    # Combine title, province, and city for text-based features
    articles_enriched['text_features'] = articles_enriched['title'] + ' ' + articles_enriched['province'] + ' ' + articles_enriched['city']
    
    # Add engagement score
    articles_enriched['engagement_score'] = articles_enriched['likes_count'] + (2 * articles_enriched['comments_count'])
    
    return articles_enriched

In [81]:
class TFContentBasedRecommender:
    def __init__(self):
        self.vocab = {}
        self.idf = {}
        self.article_embeddings = None
        self.articles_df = None
        self.model = None
        self.model_path = 'models/tf_content_recommender'

    def _tokenize(self, text):
        return text.lower().split()

    def _build_vocab(self, texts):
        vocab_counter = Counter()
        for text in texts:
            tokens = self._tokenize(text)
            vocab_counter.update(tokens)
        self.vocab = {word: idx for idx, (word, _) in enumerate(vocab_counter.items())}
    
    def _compute_idf(self, texts):
        N = len(texts)
        df = Counter()
        for text in texts:
            tokens = set(self._tokenize(text))
            df.update(tokens)
        self.idf = {word: log(N / (df[word] + 1)) for word in self.vocab}

    def _tfidf_vector(self, text):
        tokens = self._tokenize(text)
        tf = Counter(tokens)
        vector = np.zeros(len(self.vocab))
        for token in tokens:
            if token in self.vocab:
                tf_val = tf[token] / len(tokens)
                idf_val = self.idf.get(token, 0)
                vector[self.vocab[token]] = tf_val * idf_val
        return vector

    def _build_model(self, input_dim):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu', input_shape=(input_dim,)),
            tf.keras.layers.Dense(64, activation='relu')
        ])
        return model

    def fit(self, articles_df):
        self.articles_df = articles_df.copy()
        texts = self.articles_df['text_features'].tolist()

        self._build_vocab(texts)
        self._compute_idf(texts)

        tfidf_matrix = np.array([self._tfidf_vector(text) for text in texts])
        self.model = self._build_model(input_dim=tfidf_matrix.shape[1])
        self.article_embeddings = self.model.predict(tfidf_matrix)

        return self

    def recommend(self, article_id, top_n=5):
        idx = self.articles_df[self.articles_df['UUID'] == article_id].index[0]
        query_embedding = self.article_embeddings[idx].reshape(1, -1)

        similarity = np.dot(self.article_embeddings, query_embedding.T).flatten()
        norms = np.linalg.norm(self.article_embeddings, axis=1) * np.linalg.norm(query_embedding)
        similarity = similarity / norms

        sim_indices = np.argsort(similarity)[::-1][1:top_n+1]
        recommended_articles = self.articles_df.iloc[sim_indices][['UUID', 'title', 'province', 'city']].copy()
        recommended_articles['similarity_score'] = similarity[sim_indices]

        return recommended_articles

    def save_model(self):
        os.makedirs(self.model_path, exist_ok=True)
        np.save(os.path.join(self.model_path, 'article_embeddings.npy'), self.article_embeddings)
        self.articles_df.to_pickle(os.path.join(self.model_path, 'articles_df.pkl'))

        with open(os.path.join(self.model_path, 'vocab.json'), 'w') as f:
            json.dump(self.vocab, f)
        with open(os.path.join(self.model_path, 'idf.json'), 'w') as f:
            json.dump(self.idf, f)

    def load_model(self):
        if os.path.exists(self.model_path):
            self.article_embeddings = np.load(os.path.join(self.model_path, 'article_embeddings.npy'))
            self.articles_df = pd.read_pickle(os.path.join(self.model_path, 'articles_df.pkl'))

            with open(os.path.join(self.model_path, 'vocab.json'), 'r') as f:
                self.vocab = json.load(f)
            with open(os.path.join(self.model_path, 'idf.json'), 'r') as f:
                self.idf = json.load(f)

            return True
        return False

    def update_model(self, new_articles_df):
        if self.articles_df is not None:
            existing_uuids = set(self.articles_df['UUID'].values)
            new_articles = new_articles_df[~new_articles_df['UUID'].isin(existing_uuids)]

            if len(new_articles) > 0:
                self.articles_df = pd.concat([self.articles_df, new_articles]).reset_index(drop=True)
                self.fit(self.articles_df)
                self.save_model()
                return True, f"Model updated with {len(new_articles)} new articles"
            return False, "No new articles to update"
        else:
            self.fit(new_articles_df)
            self.save_model()
            return True, f"Initial model created with {len(new_articles_df)} articles"


In [71]:
 # Memuat data
articles_df, likes_df, comments_df = load_data()

In [82]:
  # Proses data
articles_enriched = preprocess_data(articles_df, likes_df, comments_df)


In [83]:
# Inisialisasi model Content-Based Recommender
recommender = TFContentBasedRecommender()

In [84]:
 # Inisialisasi dan melatih model
recommender = TFContentBasedRecommender()
recommender.fit(articles_enriched)
    
    # # Mendapatkan rekomendasi
article_id = 'a1b2c3d4-1111'  # ID artikel referensi
recommendations = recommender.recommend(article_id, top_n=3)
    
print(f"Rekomendasi untuk artikel '{articles_df[articles_df['UUID'] == article_id]['title'].values[0]}':")
for _, row in recommendations.iterrows():
    print(f"- {row['title']} (Skor: {row['similarity_score']:.4f})")


d:\dicoding\capstone\venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
Rekomendasi untuk artikel 'Wisata Pantai Kuta':
- Mengenal Tari Kecak (Skor: 0.6176)
- Mendaki Gunung Bromo (Skor: 0.5964)
- Tips Belanja di Malioboro (Skor: 0.5962)


In [85]:
    
    # Menyimpan model
recommender.save_model()

In [93]:
 # Memberikan rekomendasi artikel berdasarkan artikel dengan UUID tertentu
article_id = 'a1b2c3d4-1111'  # Ganti dengan ID artikel yang ingin dicari rekomendasinya
recommended_articles = recommender.recommend(article_id=article_id, top_n=5)

print("Artikel yang direkomendasikan:")
recommended_articles

Artikel yang direkomendasikan:


,UUID,title,province,city,similarity_score
5,a1b2c3d4-6666,Mengenal Tari Kecak,Bali,Denpasar,0.617560
2,a1b2c3d4-3333,Mendaki Gunung Bromo,Jawa Timur,Probolinggo,0.596363
4,a1b2c3d4-5555,Tips Belanja di Malioboro,DI Yogyakarta,Yogyakarta,0.596166
8,a1b2c3d4-9999,Festival Cap Go Meh,Kalimantan Barat,Singkawang,0.595992
3,a1b2c3d4-4444,Sejarah Candi Borobudur,Jawa Tengah,Magelang,0.590023
